# Student performance: initial EDA
Goal: predict the final mathematics grade (`G3`, 0-20) using information available near the start of the course. Run this notebook from any working directory; it locates the dataset relative to the project. See `docs/prediction_spec.md` for the feature decision.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

roots = [Path.cwd(), *Path.cwd().parents]
dataset = next((root / 'data/raw/student-mat.csv' for root in roots if (root / 'data/raw/student-mat.csv').is_file()), None)
if dataset is None:
    raise FileNotFoundError('Run python src/data/download.py from the project root first')
students = pd.read_csv(dataset, sep=';')
print(f'{len(students)} rows, {students.shape[1]} columns')
display(students.head())

## Quality and target distribution

In [ ]:
display(students.dtypes.value_counts().rename('columns'))
print('Missing values:', int(students.isna().sum().sum()))
print('Duplicate rows:', int(students.duplicated().sum()))
display(students['G3'].describe())
print('Zero final grades:', int((students['G3'] == 0).sum()))
students['G3'].plot.hist(bins=21, range=(-0.5, 20.5), edgecolor='white', title='Final mathematics grade (G3)')
plt.xlabel('Grade (0-20)')
plt.show()

## Relationships and prediction-time availability
Correlations are descriptive only. `G1` and `G2` are strongly related to `G3`, but they are not available at the start of the course. The full-year absence count is also unavailable then.

In [ ]:
numeric_corr = students.select_dtypes(include='number').corr()['G3'].drop('G3').sort_values(key=abs, ascending=False)
display(numeric_corr.head(12).rename('correlation_with_G3'))
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
students.boxplot(column='G3', by='studytime', ax=axes[0])
students.boxplot(column='G3', by='failures', ax=axes[1])
fig.suptitle('')
fig.tight_layout()
plt.show()

## Feature contract for the next step

In [ ]:
features = ['Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet']
target = 'G3'
assert set(features + [target]).issubset(students.columns)
assert not students[features + [target]].isna().any().any()
print(f'{len(features)} input features; target = {target}')
display(students[features + [target]].head())